In [ ]:
"""
Step 01: Price Action (PA) Snapshot Data Collection

Objective:

    - Convert every session into growing snapshots (bars 0..k) and compute the price-action columns on each snapshot
      (add_PA_cols: avg_price, atr, smooth_price, minima / maxima, extremas_pp, composite_price, slope, trend, segment).
    - The snapshots are cached per day in step01_PA_ohlcv_data (renamed from step00_PA_ohlcv_data). Steps 02 and 03 read this cache.
    - Revised columns (extremas_pp tail, composite_price, slope, trend, segment) are revised BY DESIGN as new bars
      arrive; steps 02 / 03 only keep the last rows of each snapshot, so no row uses future bars.

The logic is unchanged from the step00_label_ohlcv_data notebook of the previous workspace (constants now in config.py, same values).
Generating one day takes a few seconds; the existing cache only needs the missing dates.
"""
None

In [ ]:
# IMPORT GENERAL PACKAGES
import pandas as pd
import numpy as np
import time
# IMPORT THE SHARED CONFIGURATION
from so import config
# IMPORT TRANSFORM OHLCV DATA FUNCTIONS
from so.features.ohlcv_data_utils import format_ohlcv_pdf, add_PA_cols, process_date_PA_tf_ohlcv_pdf_list, read_date_tf_ohlcv_pdf_list
# IMPORT SNAPSHOT FEATURE FUNCTIONS
from so.features.snapshot_features import generate_date_PA_tf_ohlcv_pdf, validate_extremas_ohlcv_pdf_list
# IMPORT PLOT OHLCV FEATURES FUNCTIONS
from so.features.plot_ohlcv_utils import plot_ohlcv, plot_ohlcv_market_structure, plot_animated_ohlcv_PA
# IMPORT MARKET DATETIME FUNCTIONS
from so.core.datetime_utils import ny_tz, get_date_pdf
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import get_path_file_list, \
                                    read_pdf_from_csv_file_path_list, \
                                    generate_func_data_date_list, \
                                    get_missing_date_list
# IMPORT PATHS
from so import paths

In [ ]:
# CALL FUNCTION TO GET ALL FILES IN PATH
rawzone_data_file_path_list = get_path_file_list(paths.LOCAL_OHLCV_DATA_FILE_PATH_STR)[:]
# CALL FUNCTION TO READ PDF FROM PATH LIST
raw_ohlcv_pdf = read_pdf_from_csv_file_path_list(rawzone_data_file_path_list)
# CORRECT THE TIMESTAMP DATE COLUMN
raw_ohlcv_pdf["timestamp"] = pd.to_datetime(raw_ohlcv_pdf.timestamp, utc=True).dt.tz_convert(ny_tz)
raw_ohlcv_pdf["date"] = raw_ohlcv_pdf["timestamp"].dt.date
# CALL FUNCTION TO COLLECT DATA AND SORT BY TIMESTAMP
complete_ohlcv_pdf = format_ohlcv_pdf(raw_ohlcv_pdf.copy()).sort_values("timestamp").reset_index(drop=True)
# DISPLAY INFORMATION
print(f"min timestamp:\t{complete_ohlcv_pdf.timestamp.min()}")
print(f"max timestamp:\t{complete_ohlcv_pdf.timestamp.max()}")
# PREVIEW DATAFRAME
complete_ohlcv_pdf

In [ ]:
# DEFINE SAMPLE DATE
sample_date_str = str(complete_ohlcv_pdf["date"].iloc[-1])
# CALL FUNCTION TO GET THE PRICE ACTION SNAPSHOTS OF THE SAMPLE DATE (IN MEMORY, NOT SAVED)
start_time = time.time()
sample_PA_tf_ohlcv_pdf_list = process_date_PA_tf_ohlcv_pdf_list(complete_ohlcv_pdf, sample_date_str)
# DISPLAY INFORMATION
print(f"Snapshots:\t{len(sample_PA_tf_ohlcv_pdf_list)}\t({time.time() - start_time:.1f} seconds)")
# PREVIEW THE LAST SNAPSHOT (FULL SESSION, CONTAINS REVISED VALUES)
sample_PA_tf_ohlcv_pdf_list[-1]

In [ ]:
# CALL FUNCTION TO VALIDATE THAT CONFIRMED SWINGS NEVER CHANGE IN LATER SNAPSHOTS
validate_extremas_ohlcv_pdf_list(sample_PA_tf_ohlcv_pdf_list, alert_in=False)
# DISPLAY INFORMATION
print("✅ Confirmed minima and maxima are stable across snapshots")

In [ ]:
# CALL FUNCTION TO PLOT THE LAST SNAPSHOT
plot_ohlcv(sample_PA_tf_ohlcv_pdf_list[-1], title_str_in=sample_date_str)
# CALL FUNCTION TO PLOT THE MARKET STRUCTURE (TREND SEGMENTS)
plot_ohlcv_market_structure(sample_PA_tf_ohlcv_pdf_list[-1])

In [ ]:
# CALL FUNCTION TO ANIMATE THE FIRST 150 SNAPSHOTS (SHOWS WHICH VALUES ARE REVISED AS BARS ARRIVE)
plot_animated_ohlcv_PA(sample_PA_tf_ohlcv_pdf_list[:150])

In [ ]:
# FUNCTION: GENERATE THE DATE PRICE ACTION SNAPSHOT DATAFRAME
def generate_date_PA_tf_ohlcv_data_pdf(date_str_in):
    # CALL FUNCTION TO GENERATE THE STACKED SNAPSHOTS OF THE DATE
    return generate_date_PA_tf_ohlcv_pdf(complete_ohlcv_pdf, date_str_in)

# COLLECT THE START AND END DATE
start_date_str, end_date_str = str(complete_ohlcv_pdf["date"].min()), str(complete_ohlcv_pdf["date"].max())
# GET MISSING DATE LIST
missing_date_list = get_missing_date_list(paths.LOCAL_PA_OHLCV_DATA_FILE_PATH_STR, start_date_str, end_date_str)
# PREVIEW LIST
missing_date_list[:10]

In [ ]:
# CALL FUNCTION TO PROCESS DATE LIST
generate_func_data_date_list(generate_date_PA_tf_ohlcv_data_pdf, missing_date_list, paths.LOCAL_PA_OHLCV_DATA_FILE_PATH_STR, "ohlcv_data", "I")

In [ ]:
# CALL FUNCTION TO READ THE SAMPLE DATE FROM THE CACHE AND CHECK IT EQUALS THE IN-MEMORY SNAPSHOTS
read_PA_tf_ohlcv_pdf_list = read_date_tf_ohlcv_pdf_list(paths.LOCAL_PA_OHLCV_DATA_FILE_PATH_STR, sample_date_str)
# IF THE CACHE EXISTS
if read_PA_tf_ohlcv_pdf_list:
    # COMPARE THE LAST SNAPSHOTS AS STRINGS (CSV ROUND TRIP)
    cache_equal_bool = read_PA_tf_ohlcv_pdf_list[-1].reset_index(drop=True).drop(columns=["date"], errors="ignore").astype(str) \
                        .equals(sample_PA_tf_ohlcv_pdf_list[-1].reset_index(drop=True).drop(columns=["date"], errors="ignore").astype(str))
    # DISPLAY INFORMATION
    print("✅ Cache equals the in-memory snapshots" if cache_equal_bool else "❌ Cache differs from the in-memory snapshots")